<h3>Important Libraries</h3>

In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering,TrainingArguments,Trainer,DataCollatorWithPadding
import evaluate
import json

In [3]:
from datasets import Dataset
from datasets import DatasetDict

<h3>Load Dataset</h3>

In [4]:
def load_squad(data):
  with open(data,"r",encoding="utf-8") as f:
    squad_dict = json.load(f)

  contexts = []
  questions = []
  answers = []

  for article in squad_dict["data"]:
    for paragraph in article["paragraphs"]:
      context = paragraph["context"]

      for qa in paragraph["qas"]:
        question = qa["question"]
        answer = qa["answers"][0]

        contexts.append(context)
        questions.append(question)
        answers.append({
            "text": answer["text"],
            "answer_start": answer["answer_start"]

        })
  return contexts, questions, answers

In [5]:
train_contexts, train_questions, train_answers = load_squad('/content/train-v1.1.json')
val_contexts, val_questions, val_answers = load_squad('/content/dev-v1.1.json')

In [6]:
print("Train samples: ", len(train_contexts))
print("Validation samples: ", len(val_contexts))

Train samples:  87599
Validation samples:  10570


<h3>Convert to hugging face dataset format</h3>

In [7]:
def create_dataset(contexts, questions, answers):
  formatted_answers = []

  for ans in answers:
    formatted_answers.append({
        "text":[ans["text"]],
        "answer_start": [ans["answer_start"]]
    })

  return Dataset.from_dict({
      "context": contexts,
      "question": questions,
      "answers": formatted_answers
  })

train_dataset = create_dataset(train_contexts, train_questions,train_answers)
val_dataset = create_dataset(val_contexts, val_questions, val_answers)

In [8]:
print(train_dataset)

Dataset({
    features: ['context', 'question', 'answers'],
    num_rows: 87599
})


In [9]:
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

<h3>Tokenization</h3>

In [10]:
model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

<h3>Preprocessing</h3>

In [11]:
max_length = 384
doc_stride = 128

def preprocess(text):
  question = [q.strip() for q in text["question"]]

  inputs = tokenizer(
      question,
      text["context"],
      max_length=max_length,
      truncation="only_second",
      return_offsets_mapping=True,
      return_overflowing_tokens=True,
      stride=doc_stride
  )

  offset_mapping = inputs.pop("offset_mapping")
  sample_mapping = inputs.pop("overflow_to_sample_mapping")

  start_positions = []
  end_positions = []

  for i, offsets in enumerate(offset_mapping):
    input_ids = inputs["input_ids"][i]
    cls_index = input_ids.index(tokenizer.cls_token_id)

    sequence_ids = inputs.sequence_ids(i)
    sample_index = sample_mapping[i]
    answers = text["answers"][sample_index]

    if len(answers["answer_start"]) == 0:
      start_positions.append(cls_index)
      end_positions.append(cls_index)

    else:
      start_char = answers["answer_start"][0]
      end_char = start_char + len(answers["text"][0]) # Corrected line

      token_start_index = 0
      while sequence_ids[token_start_index] != 1:
        token_start_index+=1

      token_end_index = len(input_ids) - 1
      while sequence_ids[token_end_index] != 1:
        token_end_index-=1

      if not (offsets[token_start_index][0] <= start_char and
              offsets[token_end_index][1] >= end_char):
        start_positions.append(cls_index)
        end_positions.append(cls_index)
      else:
        while token_start_index < len(offsets) and offsets[token_start_index][0]<= start_char:
          token_start_index+=1
        start_positions.append(token_start_index - 1)

        while offsets[token_end_index][1] >= end_char:
          token_end_index -=1
        end_positions.append(token_end_index + 1)

  inputs["start_positions"] = start_positions
  inputs["end_positions"] = end_positions

  return inputs

In [12]:
tokenized_datasets = dataset.map(
    preprocess,
    batched = True,
    remove_columns = dataset["train"].column_names
)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

<h3>Load Model</h3>

In [13]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [14]:
training_args = TrainingArguments(
    output_dir = "/content/results",
    learning_rate = 2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit = 2,
    push_to_hub=False
)

In [15]:
metric = evaluate.load("squad")

In [16]:
def compute_metrics(eval_pred):
  return metric.compute(predictions=eval_pred.predictions,references=eval_pred.label_ids)

<h3>Initialize Trainer</h3>

In [17]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["validation"],
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics = compute_metrics
)

<h3>Train Model</h3>

In [ ]:
trainer.train()

Step,Training Loss
500,2.677212
1000,1.650343
1500,1.481222
2000,1.372930
2500,1.330733
3000,1.312045
3500,1.315730
4000,1.252152
4500,1.272031
5000,1.212272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=22132, training_loss=1.0126747439529415, metrics={'train_runtime': 2170.0162, 'train_samples_per_second': 81.588, 'train_steps_per_second': 10.199, 'total_flos': 2.4664245889349856e+16, 'train_loss': 1.0126747439529415, 'epoch': 2.0})